# Pilas · Modelado de riesgo con datos reales (Cali)

Notebook de la Fase 2 del PLAN. Usa los datos reales de `Bases de datos/` (hoja «TAB ALCALDÍA 09-19», ~170k incidentes 2010–2019) para explorar patrones espacio-temporales y entrenar un XGBoost (Poisson) que estima incidentes por **comuna × hora × día × mes**.

Ejecútalo desde `backend/` (para que `import app` e `import ml` funcionen).

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'ml':
    sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import matplotlib.pyplot as plt
from app import config, data

## 1. Ingesta de las bases reales

Genera los CSVs limpios (`incidents_cali.csv`, etc.). Solo hace falta correrlo una vez.

In [ ]:
from ml.ingest import main as ingest_main
if not (config.DATA_DIR / 'incidents_cali.csv').exists():
    ingest_main()
df = pd.read_csv(config.DATA_DIR / 'incidents_cali.csv')
print(df.shape)
df.head()

## 2. Análisis exploratorio (EDA)

Patrón horario y por comuna sobre datos reales — el hurto a personas en Cali pica a mediodía/tarde, no de madrugada.

In [ ]:
df.groupby('hour')['count'].sum().plot(kind='bar', figsize=(10, 3), title='Incidentes por hora (2010–2019)')
plt.tight_layout(); plt.show()

In [ ]:
by_comuna = df.groupby('comuna')['count'].sum().sort_values(ascending=False)
by_comuna.plot(kind='bar', figsize=(10, 3), title='Incidentes por comuna')
plt.tight_layout(); plt.show()
by_comuna.head(8)

In [ ]:
# Mapa de calor comuna × hora (riesgo relativo)
pivot = df.pivot_table(index='comuna', columns='hour', values='count', aggfunc='sum', fill_value=0)
plt.figure(figsize=(12, 6))
plt.imshow(pivot, aspect='auto', cmap='magma')
plt.colorbar(label='incidentes'); plt.xlabel('hora'); plt.ylabel('comuna (idx)')
plt.title('Incidentes por comuna × hora'); plt.tight_layout(); plt.show()

## 3. Entrenamiento

Reutiliza el pipeline de `ml/train.py`: rellena ceros, split temporal (2010–2017 train / 2018 test), métricas y guardado del artefacto.

In [ ]:
from ml.train import main as train_main
train_main()

## 4. Explicabilidad e inferencia

Importancia de features (gain) y ejemplo de scoring 0–100 por zona/hora.

In [ ]:
import json
meta = json.loads(config.METRICS_PATH.read_text(encoding='utf-8'))
print('Métricas:', meta['metrics'])
imp = pd.Series(meta['feature_importance']).sort_values()
imp.plot(kind='barh', figsize=(7, 3), title='Importancia de features (gain)')
plt.tight_layout(); plt.show()

In [ ]:
from app.model import risk_model
risk_model.load()
for zid in ('pance', 'centro', 'aguablanca', 'granada'):
    z = data.get_zone(zid)
    print(z['name'], {h: risk_model.score(z, h)['risk'] for h in (3, 8, 14, 19, 22)})